# 5 類災害影像 EDA（Kaggle，Run All）

針對 `train_5class_disaster_kaggle.ipynb` 使用的資料（HF `QCRI/MEDIC`，保留 5 類真實災害：地震 / 淹水 / 火災 / 颱風 / 土石流）做探索性分析。所有圖表會存成 `/kaggle/working/eda_*.png`，跑完後下載、貼回對話，再依圖表討論模型優化方向。

## 使用前
1. 開 **Internet**；建議開 **GPU**（只有 §7 特徵抽取會用到，沒有也能跑，只是較慢）。
2. （可選）Add-ons → Secrets 設 `HF_TOKEN` 加速下載。
3. **Run All**。全程約 15–30 分鐘（多數時間在下載資料與 §3 全量掃描）。

## 章節
| 節 | 內容 | 產出 |
|---|---|---|
| §1 | 類別分布與不平衡 | eda_01 |
| §2 | 多任務標籤交叉（informative / damage_severity / humanitarian） | eda_02 |
| §3 | 影像尺寸 / 格式 / 檔案大小（全量掃描） | eda_03 |
| §4 | 每類隨機樣本目視 | eda_04 |
| §5 | 亮度 / 飽和度 / RGB 色彩特徵 | eda_05a, eda_05b |
| §6 | 重複影像與 train/test 洩漏 | eda_06 |
| §7 | 特徵空間可分性（t-SNE + kNN baseline） | eda_07, eda_08 |
| §8 | 訓練增強預覽 | eda_09 |
| §9 | 自動彙整 + 下載連結 | eda_findings.json |

> 圖表內文字一律用英文：Kaggle 預設無中文字型，圖中中文會變成方框。

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
HF_DATASET = "QCRI/MEDIC"

# ── 5 類純災害（與 train_5class_disaster_kaggle.ipynb 的 MEDIC_MAP 一致）──
CLASSES_EN  = ["Earthquake", "Flood", "Fire", "Hurricane", "Landslide"]
MEDIC_MAP   = {"earthquake": 0, "flood": 1, "fire": 2, "hurricane": 3, "landslide": 4}
NUM_CLASSES = 5

SEED = 42
SAMPLE_PIXEL_PER_CLASS = 300   # §5 像素統計：train 每類抽樣張數
EMBED_TRAIN_PER_CLASS  = 300   # §7 特徵抽取：train 每類抽樣張數
EMBED_TEST_PER_CLASS   = 200   # §7 特徵抽取：test 每類抽樣張數
GRID_PER_CLASS         = 8     # §4 每類展示張數
TSNE_PERPLEXITY        = 30
FIG_DPI                = 120

In [ ]:
import os, io, json, time, glob, hashlib, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
import torchvision.models as tvm
from datasets import load_dataset, ClassLabel, Image as HFImage
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

FIG_DIR  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
FINDINGS = {}   # 各節自動累積重點數字，§9 彙整存 json

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    print("saved:", path)

In [ ]:
# （可選）HF 認證
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("已以 HF_TOKEN 認證")
except Exception as e:
    print(f"未使用 HF_TOKEN（{type(e).__name__}），改未認證下載")

raw = load_dataset(HF_DATASET)
print(raw)
SPLIT_TRAIN = "train"
SPLIT_DEV   = "validation" if "validation" in raw else ("dev" if "dev" in raw else None)
SPLIT_TEST  = "test" if "test" in raw else None
assert SPLIT_DEV and SPLIT_TEST, f"找不到 dev/test split：{list(raw.keys())}"
SPLITS = [("train", SPLIT_TRAIN), ("val", SPLIT_DEV), ("test", SPLIT_TEST)]
tr = raw[SPLIT_TRAIN]

feats = tr.features
DT_NAMES = feats["disaster_types"].names
EXTRA_LABEL_COLS = [c for c, f in feats.items()
                    if isinstance(f, ClassLabel) and c != "disaster_types"]
print("disaster_types:", DT_NAMES)
print("其他標籤欄:", EXTRA_LABEL_COLS)
print("所有欄位:", list(feats.keys()))

## §1 類別分布（eda_01）

看三件事：
1. **不平衡比**（train 最大類 / 最小類）— 決定 sampler / class weight / 評估指標（macro-F1 而非 accuracy）。
2. **三個 split 的類別占比是否一致** — 若 test 分布偏離 train，test 分數會與實際表現脫節。
3. **最小類的絕對張數** — 太少的類靠增強撐不起來，可能需要外部資料或合併類別。

In [ ]:
name2idx5 = {n: MEDIC_MAP.get(n, -1) for n in DT_NAMES}
labels5, kept_idx = {}, {}
for tag, sp in SPLITS:
    dt = np.array(raw[sp]["disaster_types"])
    y5 = np.array([name2idx5[DT_NAMES[v]] for v in dt])
    labels5[tag] = y5
    kept_idx[tag] = np.where(y5 >= 0)[0]
    print(f"{tag:5s}: 全部 {len(y5):6d} | 保留 5 類 {len(kept_idx[tag]):6d} | 排除 not/other {int((y5 < 0).sum()):6d}")

cnt = pd.DataFrame({tag: [int((labels5[tag] == c).sum()) for c in range(NUM_CLASSES)]
                    for tag, _ in SPLITS}, index=CLASSES_EN)
cnt["train_pct"] = (cnt["train"] / cnt["train"].sum() * 100).round(1)
imb = cnt["train"].max() / cnt["train"].min()
FINDINGS["counts"] = cnt[["train", "val", "test"]].to_dict()
FINDINGS["imbalance_ratio_train"] = round(float(imb), 2)
print(); print(cnt.to_string())
print(f"\nTrain 不平衡比（最大/最小）= {imb:.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cnt[["train", "val", "test"]].plot(kind="bar", ax=axes[0])
axes[0].set_title(f"Class counts per split (train imbalance ratio = {imb:.1f}x)")
axes[0].set_ylabel("images"); axes[0].tick_params(axis="x", rotation=20)
for c_ in axes[0].containers:
    axes[0].bar_label(c_, fontsize=7)

share = cnt[["train", "val", "test"]] / cnt[["train", "val", "test"]].sum(axis=0)
share.T.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab10")
axes[1].set_title("Class share within each split (should look similar)")
axes[1].set_ylabel("share")
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout(); savefig("eda_01_class_distribution.png"); plt.show()

## §2 多任務標籤交叉（eda_02）

MEDIC 每張圖另有三個任務標籤：`informative`、`damage_severity`、`humanitarian`。
交叉表可看出各災害類的「雜訊含量」：例如某類大量 `not_informative` 或 `little_or_none`（幾乎無災損畫面）——這些圖標籤上屬於該災害事件、畫面上卻沒有災害特徵，是訓練雜訊的主要來源，也是之後資料清理的候選對象。

In [ ]:
if EXTRA_LABEL_COLS:
    y5 = labels5["train"]
    fig, axes = plt.subplots(1, len(EXTRA_LABEL_COLS), figsize=(6 * len(EXTRA_LABEL_COLS), 5))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, EXTRA_LABEL_COLS):
        names = tr.features[col].names
        v = np.array(tr[col])
        ct = pd.DataFrame(0.0, index=CLASSES_EN, columns=names)
        for c in range(NUM_CLASSES):
            vc = v[y5 == c]
            for j, n in enumerate(names):
                ct.loc[CLASSES_EN[c], n] = float((vc == j).mean())
        ct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
        ax.set_title(f"disaster type x {col} (train, row share)")
        ax.tick_params(axis="x", rotation=20)
        ax.legend(fontsize=7, loc="upper right")
        FINDINGS[f"crosstab_{col}"] = ct.round(3).to_dict()
    plt.tight_layout(); savefig("eda_02_label_crosstab.png"); plt.show()
else:
    print("沒有其他 ClassLabel 標籤欄，略過本節")

## §3 全量掃描：尺寸 / 格式 / 雜湊（eda_03）

走訪三個 split 的所有保留影像。MD5 與影像尺寸只讀位元組與檔頭；dHash 用 JPEG draft 低解析度解碼，整體約幾分鐘。
- 尺寸過小（min side < 224）的影像經 `RandomResizedCrop(224)` 會被放大、變模糊
- MD5 / dHash 供 §6 重複與洩漏分析

In [ ]:
def dhash64(img_bytes):
    """8x8 dHash；JPEG 用 draft 低解析解碼加速。回傳 16 字元 hex。"""
    with Image.open(io.BytesIO(img_bytes)) as im:
        im.draft("L", (64, 64))
        g = im.convert("L").resize((9, 8), Image.BILINEAR)
        px_ = np.asarray(g, dtype=np.int16)
    bits = (px_[:, 1:] > px_[:, :-1]).flatten()
    return np.packbits(bits).tobytes().hex()

rows, t0 = [], time.time()
for tag, sp in SPLITS:
    ds_nd = raw[sp].cast_column("image", HFImage(decode=False))
    idxs  = kept_idx[tag]
    y_arr = labels5[tag]
    sub   = ds_nd.select([int(i) for i in idxs])
    for k, item in enumerate(sub):
        b = item["image"]["bytes"]
        if b is None:
            with open(item["image"]["path"], "rb") as f:
                b = f.read()
        try:
            with Image.open(io.BytesIO(b)) as im:
                w, h = im.size; fmt = im.format; mode = im.mode
            dh = dhash64(b)
        except Exception:
            w = h = -1; fmt = mode = "ERR"; dh = ""
        rows.append(dict(split=tag, hf_i=int(idxs[k]), y=int(y_arr[idxs[k]]),
                         w=w, h=h, fmt=fmt, mode=mode, kb=len(b) / 1024,
                         md5=hashlib.md5(b).hexdigest(), dhash=dh))
        if (k + 1) % 3000 == 0:
            print(f"  {tag}: {k + 1}/{len(idxs)} ({time.time() - t0:.0f}s)")
    print(f"{tag} 完成：{len(idxs)} 張（{time.time() - t0:.0f}s）")

meta = pd.DataFrame(rows)
meta["cls"]      = meta["y"].map(dict(enumerate(CLASSES_EN)))
meta["min_side"] = meta[["w", "h"]].min(axis=1)
meta["aspect"]   = meta["w"] / meta["h"]
meta_ok = meta[meta["w"] > 0]
FINDINGS["unreadable_images"] = int((meta["fmt"] == "ERR").sum())
print(meta.shape, "| 無法讀取:", FINDINGS["unreadable_images"])
meta.head()

In [ ]:
pct_tiny = float((meta_ok["min_side"] < 224).mean() * 100)
FINDINGS["pct_min_side_lt_224"] = round(pct_tiny, 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.histplot(data=meta_ok, x="min_side", hue="split", bins=60, element="step",
             common_norm=False, stat="percent", ax=axes[0])
axes[0].axvline(224, color="red", ls="--", lw=1)
axes[0].set_xlim(0, 1200)
axes[0].set_title(f"Min side distribution ({pct_tiny:.1f}% < 224, red line)")
axes[0].set_xlabel("min(w, h) px (view <= 1200)")

sns.boxplot(data=meta_ok[meta_ok["split"] == "train"], x="cls", y="min_side",
            order=CLASSES_EN, showfliers=False, ax=axes[1])
axes[1].axhline(224, color="red", ls="--", lw=1)
axes[1].set_title("Min side by class (train, no outliers)")
axes[1].tick_params(axis="x", rotation=20)

sns.histplot(meta_ok["aspect"].clip(0.2, 5), bins=60, ax=axes[2])
axes[2].set_title("Aspect ratio w/h (clipped to [0.2, 5])")
plt.tight_layout(); savefig("eda_03_image_size.png"); plt.show()

print(meta_ok.groupby("cls")[["w", "h", "min_side", "kb"]].median().round(0).reindex(CLASSES_EN).to_string())
print("\n格式分布："); print(meta["fmt"].value_counts().to_string())
print("\n色彩模式："); print(meta["mode"].value_counts().to_string())

## §4 每類隨機樣本目視（eda_04）

社群媒體影像常混入：新聞畫面截圖、拼貼圖、地圖、純文字圖。目視檢查每類的標籤雜訊與視角組成（空拍 vs 地面）——這類雜訊圖是 CNN 學歪的常見原因。要換一批樣本可改 `SEED` 重跑。

In [ ]:
fig, axes = plt.subplots(NUM_CLASSES, GRID_PER_CLASS,
                         figsize=(2.2 * GRID_PER_CLASS, 2.5 * NUM_CLASSES))
for c in range(NUM_CLASSES):
    pool = np.where(labels5["train"] == c)[0]
    pick = rng.choice(pool, size=min(GRID_PER_CLASS, len(pool)), replace=False)
    for j in range(GRID_PER_CLASS):
        ax = axes[c, j]; ax.axis("off")
        if j < len(pick):
            im = tr[int(pick[j])]["image"].convert("RGB")
            im.thumbnail((224, 224))
            ax.imshow(im)
    axes[c, 0].set_title(CLASSES_EN[c], loc="left", fontsize=11, color="navy")
plt.tight_layout(); savefig("eda_04_samples_grid.png"); plt.show()

## §5 像素統計（eda_05a / eda_05b）

每類抽 `SAMPLE_PIXEL_PER_CLASS` 張，計算亮度 / 對比 / 飽和度 / RGB 通道平均：
- 各類顏色特徵差距大（直覺上：火災偏紅暖、淹水偏棕）→ 顏色本身是判別線索，`ColorJitter` 過強會把線索抹掉
- 資料集通道 mean 與 ImageNet 預設值差多少 → 決定要不要改 Normalize 參數（用預訓練骨幹時通常保留 ImageNet 值）

In [ ]:
recs = []
for c in range(NUM_CLASSES):
    pool = np.where(labels5["train"] == c)[0]
    pick = rng.choice(pool, size=min(SAMPLE_PIXEL_PER_CLASS, len(pool)), replace=False)
    for i in pick:
        im = tr[int(i)]["image"].convert("RGB"); im.thumbnail((128, 128))
        a   = np.asarray(im, dtype=np.float32) / 255.0
        hsv = np.asarray(im.convert("HSV"), dtype=np.float32) / 255.0
        recs.append(dict(cls=CLASSES_EN[c], brightness=float(a.mean()),
                         contrast=float(a.std()), saturation=float(hsv[..., 1].mean()),
                         r=float(a[..., 0].mean()), g=float(a[..., 1].mean()), b=float(a[..., 2].mean())))
px = pd.DataFrame(recs)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, colname, title in zip(axes, ["brightness", "saturation", "contrast"],
                              ["Brightness", "Saturation", "Contrast (pixel std)"]):
    sns.violinplot(data=px, x="cls", y=colname, order=CLASSES_EN, cut=0, ax=ax)
    ax.set_title(f"{title} by class (train, {SAMPLE_PIXEL_PER_CLASS}/class)")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); savefig("eda_05a_pixel_stats.png"); plt.show()

rgb = px.groupby("cls")[["r", "g", "b"]].mean().reindex(CLASSES_EN)
ax = rgb.plot(kind="bar", color=["tab:red", "tab:green", "tab:blue"], figsize=(8, 4))
ax.set_title("Mean RGB by class (color signature)")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); savefig("eda_05b_rgb_by_class.png"); plt.show()

ds_mean = [round(float(v), 3) for v in px[["r", "g", "b"]].mean()]
FINDINGS["channel_mean_sampled"] = ds_mean
print("抽樣通道 mean:", ds_mean, "  vs ImageNet [0.485, 0.456, 0.406]")

## §6 重複影像與 train/test 洩漏（eda_06）

社群媒體同一張圖常被多帳號轉發。若近乎相同的圖同時出現在 train 與 test，**test 分數會虛高**，模型「背圖」也看不出來。
- MD5 相同 = 位元組完全相同；dHash 相同 = 視覺上同一張（經縮放 / 重壓縮）
- 右側並排展示最多 3 組 train/test 重複對，確認不是誤判

In [ ]:
def dup_stats(key):
    df = meta[meta[key] != ""]
    res = {}
    for tag, _ in SPLITS:
        res[f"within_{tag}"] = int(df.loc[df["split"] == tag, key].duplicated().sum())
    train_set = set(df.loc[df["split"] == "train", key])
    for tag in ["val", "test"]:
        m = df["split"] == tag
        res[f"{tag}_in_train"] = int(df.loc[m, key].isin(train_set).sum())
    return res

md5_s, dh_s = dup_stats("md5"), dup_stats("dhash")
dup_df = pd.DataFrame({"md5 (exact)": md5_s, "dhash (near)": dh_s})
print(dup_df.to_string())

train_dh = set(meta.loc[(meta["split"] == "train") & (meta["dhash"] != ""), "dhash"])
leak = meta[(meta["split"] == "test") & meta["dhash"].isin(train_dh)]
leak_pct = float(len(leak) / (meta["split"] == "test").sum() * 100)
FINDINGS["dup_md5"] = md5_s
FINDINGS["dup_dhash"] = dh_s
FINDINGS["test_leak_pct_dhash"] = round(leak_pct, 2)
FINDINGS["test_leak_by_class"] = leak["cls"].value_counts().to_dict()
print(f"\nTest 中與 train dHash 重複：{len(leak)} 張（= {leak_pct:.2f}% of test）")
print(leak["cls"].value_counts().to_string() if len(leak) else "（無）")

fig = plt.figure(figsize=(14, 5))
ax0 = plt.subplot2grid((2, 5), (0, 0), rowspan=2, colspan=2)
dup_df.plot(kind="bar", ax=ax0, legend=True)
ax0.set_title(f"Duplicate counts (test leak = {leak_pct:.2f}% by dHash)")
ax0.tick_params(axis="x", rotation=20)
if len(leak) > 0:
    tr_by_dh = (meta[(meta["split"] == "train") & (meta["dhash"] != "")]
                .drop_duplicates("dhash").set_index("dhash"))
    for r, (_, row) in enumerate(leak.head(3).iterrows()):
        im_tr = tr[int(tr_by_dh.loc[row["dhash"], "hf_i"])]["image"].convert("RGB")
        im_te = raw[SPLIT_TEST][int(row["hf_i"])]["image"].convert("RGB")
        for rr, (im, t) in enumerate([(im_tr, "train"), (im_te, "test")]):
            im.thumbnail((180, 180))
            ax = plt.subplot2grid((2, 5), (rr, 2 + r))
            ax.imshow(im); ax.axis("off")
            ax.set_title(f"{t} ({row['cls']})", fontsize=8)
plt.tight_layout(); savefig("eda_06_duplicates.png"); plt.show()

## §7 特徵空間可分性：t-SNE + kNN（eda_07 / eda_08）

用 **未微調** 的 ImageNet ResNet18 抽 512 維特徵：
- **t-SNE**：5 類在通用視覺特徵下的重疊程度；重疊最嚴重的類對，就是微調後混淆矩陣裡最花的格子
- **kNN（k=5, cosine）**：免訓練 baseline。微調模型若沒有明顯贏過它，代表訓練流程（而非資料）有問題

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
emb_net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
emb_net.fc = nn.Identity()
emb_net = emb_net.eval().to(device)
emb_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

@torch.no_grad()
def embed_samples(hf_ds, y_all, per_class):
    out, ys, batch = [], [], []
    for c in range(NUM_CLASSES):
        pool = np.where(y_all == c)[0]
        pick = rng.choice(pool, size=min(per_class, len(pool)), replace=False)
        for i in pick:
            batch.append(emb_tf(hf_ds[int(i)]["image"].convert("RGB")))
            ys.append(c)
            if len(batch) == 64:
                out.append(emb_net(torch.stack(batch).to(device)).cpu().numpy())
                batch = []
    if batch:
        out.append(emb_net(torch.stack(batch).to(device)).cpu().numpy())
    return np.concatenate(out), np.array(ys)

t0 = time.time()
Xtr, ytr = embed_samples(tr, labels5["train"], EMBED_TRAIN_PER_CLASS)
Xte, yte = embed_samples(raw[SPLIT_TEST], labels5["test"], EMBED_TEST_PER_CLASS)
print(f"embeddings: train {Xtr.shape} / test {Xte.shape}（{time.time() - t0:.0f}s）")

XY = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY, init="pca",
          learning_rate="auto", random_state=SEED).fit_transform(np.vstack([Xtr, Xte]))
ntr = len(Xtr)
palette = sns.color_palette("tab10", NUM_CLASSES)
plt.figure(figsize=(9, 7))
for c in range(NUM_CLASSES):
    m = ytr == c
    plt.scatter(XY[:ntr][m, 0], XY[:ntr][m, 1], s=8, alpha=0.5,
                color=palette[c], label=CLASSES_EN[c])
    m2 = yte == c
    plt.scatter(XY[ntr:][m2, 0], XY[ntr:][m2, 1], s=16, alpha=0.85,
                color=palette[c], marker="x")
plt.legend(fontsize=9, markerscale=2)
plt.title("t-SNE of ImageNet ResNet18 features (o = train, x = test)")
plt.tight_layout(); savefig("eda_07_tsne.png"); plt.show()

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, metric="cosine").fit(Xtr, ytr)
pred = knn.predict(Xte)
acc = float((pred == yte).mean())
FINDINGS["knn5_cosine_acc"] = round(acc, 4)
print(f"kNN（k=5, cosine）test 抽樣 acc = {acc:.4f}\n")
print(classification_report(yte, pred, target_names=CLASSES_EN, digits=3))

cm = confusion_matrix(yte, pred, normalize="true")
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES_EN, yticklabels=CLASSES_EN)
plt.title(f"kNN-on-features confusion (row-normalized, acc = {acc:.3f})")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(rotation=30)
plt.tight_layout(); savefig("eda_08_knn_confusion.png"); plt.show()

## §8 訓練增強預覽（eda_09）

把訓練 notebook 的增強配方（RandomResizedCrop + HFlip + ColorJitter 0.3 + Rotation 15°）套在每類一張圖上各 5 次。
檢查：裁切後災害主體是否常被切掉？ColorJitter 後火災是否還像火災、淹水是否還像淹水？

In [ ]:
aug = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
])
N_AUG = 5
fig, axes = plt.subplots(NUM_CLASSES, N_AUG + 1,
                         figsize=(2.2 * (N_AUG + 1), 2.5 * NUM_CLASSES))
for c in range(NUM_CLASSES):
    pool = np.where(labels5["train"] == c)[0]
    im0 = tr[int(rng.choice(pool))]["image"].convert("RGB")
    show0 = im0.copy(); show0.thumbnail((224, 224))
    axes[c, 0].imshow(show0); axes[c, 0].axis("off")
    axes[c, 0].set_title(f"{CLASSES_EN[c]} (orig)", fontsize=9, color="navy")
    for j in range(1, N_AUG + 1):
        axes[c, j].imshow(aug(im0)); axes[c, j].axis("off")
        if c == 0:
            axes[c, j].set_title(f"aug {j}", fontsize=9)
plt.tight_layout(); savefig("eda_09_aug_preview.png"); plt.show()

## §9 自動彙整與下載

數字彙整存 `eda_findings.json`；所有圖表連結列在下方。互動 session 結束會清空 `/kaggle/working`，記得當下下載或 **Save Version**。

In [ ]:
from IPython.display import FileLink, display

print("=" * 58)
print("EDA SUMMARY")
print("=" * 58)
print(f"Train 不平衡比          : {FINDINGS.get('imbalance_ratio_train')}x")
print(f"min side < 224 比例     : {FINDINGS.get('pct_min_side_lt_224')}%")
print(f"無法讀取影像            : {FINDINGS.get('unreadable_images')}")
print(f"Test dHash 洩漏比例     : {FINDINGS.get('test_leak_pct_dhash')}%")
print(f"kNN(k=5) 特徵 baseline  : {FINDINGS.get('knn5_cosine_acc')}")
print(f"抽樣通道 mean           : {FINDINGS.get('channel_mean_sampled')}（ImageNet [0.485, 0.456, 0.406]）")

with open(os.path.join(FIG_DIR, "eda_findings.json"), "w", encoding="utf-8") as f:
    json.dump(FINDINGS, f, ensure_ascii=False, indent=2, default=str)
print("\n下載連結：")
for p in sorted(glob.glob(os.path.join(FIG_DIR, "eda_*"))):
    display(FileLink(p))

## 下一步：把圖貼回對話討論優化

| 圖 | 看什麼 | 對應的優化決策 |
|---|---|---|
| eda_01 類別分布 | 不平衡比、split 間占比一致性、最小類張數 | sampler / class weight / macro-F1；最小類補資料或合併 |
| eda_02 標籤交叉 | 各類 not_informative、little_or_none 占比 | 是否先過濾低資訊影像再訓練 |
| eda_03 尺寸 | <224 占比、極端長寬比 | 輸入解析度、RandomResizedCrop 的 scale 範圍 |
| eda_04 樣本網格 | 標籤雜訊、截圖 / 拼貼 / 地圖類影像 | 資料清理策略 |
| eda_05a/05b 像素統計 | 各類顏色特徵差距 | ColorJitter 強度（顏色是線索就調小）、Normalize 參數 |
| eda_06 重複/洩漏 | test 洩漏比例與類別 | 先去重再評估，否則 test 分數虛高 |
| eda_07 t-SNE | 哪些類對重疊 | 難分類對 → 針對性增強 / 階層式分類 / 合併 |
| eda_08 kNN 混淆 | 免訓練 baseline 分數與混淆熱點 | 微調模型的「最低標」；混淆對 = 優化重點 |
| eda_09 增強預覽 | 增強後語意是否保留 | 調整增強配方 |

**最少貼回這幾張**：eda_01、eda_06、eda_07、eda_08，再加上 `eda_findings.json` 的內容（直接貼文字即可），其餘視情況補充。